# Metabolite GPrediction Colab Workflow

This notebook clones the repository, installs dependencies, trains a SELFIES model with MLflow logging, evaluates generation quality, and runs sample inference without manual shell command editing.

In [ ]:
!git clone https://github.com/Gtedget/Metabolite_gprediction.git
%cd /content/Metabolite_gprediction
!pip install -r requirements.txt

In [ ]:
from datetime import datetime
from pathlib import Path

USE_GOOGLE_DRIVE_FOR_MLFLOW = False
if USE_GOOGLE_DRIVE_FOR_MLFLOW:
    from google.colab import drive
    drive.mount('/content/drive')
    MLFLOW_URI = 'file:/content/drive/MyDrive/metabolite_mlruns'
else:
    MLFLOW_URI = 'file:/content/mlruns'

RUN_NAME = datetime.utcnow().strftime('colab_%Y%m%d_%H%M%S')
ARTIFACT_DIR = Path('artifacts') / RUN_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('MLflow URI:', MLFLOW_URI)
print('Run name:', RUN_NAME)
print('Artifact dir:', ARTIFACT_DIR)

In [ ]:
import subprocess

train_cmd = [
    'python', 'train.py',
    '--data', 'train.csv',
    '--epochs', '20',
    '--batch_size', '16',
    '--lr', '1e-4',
    '--device', 'cuda',
    '--representation', 'selfies',
    '--output_dir', 'artifacts',
    '--run_name', RUN_NAME,
    '--use_mlflow',
    '--mlflow_experiment', 'metabolite-predictor-colab',
    '--mlflow_run_name', RUN_NAME,
    '--mlflow_tracking_uri', MLFLOW_URI,
]
subprocess.run(train_cmd, check=True)
MODEL_PATH = str(ARTIFACT_DIR / 'trained_model.best.pt')
METADATA_PATH = str(ARTIFACT_DIR / 'trained_model.metadata.json')
print('Best checkpoint:', MODEL_PATH)
print('Metadata:', METADATA_PATH)

In [ ]:
eval_cmd = [
    'python', 'evaluate_generation.py',
    '--data', 'test.csv',
    '--model', MODEL_PATH,
    '--metadata', METADATA_PATH,
    '--top_k', '5',
    '--beam_width', '5',
    '--device', 'cuda',
    '--out', str(ARTIFACT_DIR / 'generation_eval.json'),
    '--use_mlflow',
    '--mlflow_experiment', 'metabolite-generation-eval-colab',
    '--mlflow_run_name', f'{RUN_NAME}-eval',
    '--mlflow_tracking_uri', MLFLOW_URI,
]
subprocess.run(eval_cmd, check=True)

In [ ]:
from inference import _resolve_device, beam_search_candidates, load_model

device = _resolve_device('cuda')
model, tokenizer, metadata = load_model(MODEL_PATH, METADATA_PATH, device)
precursor_smiles = 'CC(=O)Oc1ccccc1C(=O)O'
candidates, transform_predictions = beam_search_candidates(
    model,
    tokenizer,
    precursor_smiles,
    device,
    num_candidates=5,
    beam_width=5,
)
print('Transformation predictions:', transform_predictions[:5])
candidates[:5]